In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 判斷使用者輸入是否以 "AI " 開頭
        # 只有以 "AI " 開頭的訊息才會送給 Gemini 處理
        if text.startswith('AI '):

            # 移除前面的 "AI "，只留下真正要問 Gemini 的問題
            # 例如使用者輸入 "AI 明新科技大學校長是誰"
            # prompt 會變成 "明新科技大學校長是誰"
            prompt = text[3:]

            # 使用 stateful_query() 呼叫 Gemini 產生回應
            #
            # 為什麼校長資訊可能會是比較新的？
            #
            # 因為程式中沒有把「校長姓名」固定寫死。
            # 也就是說，程式不是直接寫：
            # reply_text = "校長是某某某"
            #
            # 而是把使用者的問題 prompt 傳給 Gemini，
            # 讓 Gemini 根據目前可用的資料來產生回答。
            #
            # 這樣的好處是：
            # 如果校長之後更換，程式本身不一定需要修改固定文字，
            # Gemini 可以依照它可取得的資料內容回答。
            #
            # 例如：
            # 使用者輸入：AI 明新科技大學校長是誰
            #
            # 程式會把「明新科技大學校長是誰」傳給 Gemini，
            # Gemini 再根據資料判斷目前的校長是誰。
            #
            # 但是要特別注意：
            # stateful_query() 代表 Gemini 可以保留前後文，
            # 讓對話可以接續前面的內容。
            #
            # stateful 不代表資料一定永遠最新。
            #
            # 如果要確保「校長一定是最新的」，
            # Gemini 必須有連接最新資料來源，
            # 或者另外串接學校官方網站、搜尋 API、資料庫等即時資料。
            #
            # 因此本程式的設計是：
            # 不把校長姓名寫死在程式中，
            # 而是交由 Gemini 根據可用資料回答，
            # 讓回答比固定寫死的答案更有彈性。
            #
            # 若要作為正式資料，仍應以明新科技大學官方網站公告為準。
            reply_text = stateful_query(prompt)

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            # 如果使用者輸入不是以 "AI " 開頭
            # 就不會呼叫 Gemini，而是直接回覆原本輸入的文字
            #
            # 這裡放了兩個 TextMessage，
            # 所以 LINE Bot 會回覆兩次相同內容
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


if __name__ == "__main__":
    app.run(port=port)